# Setup

Import library yang diperlukan untuk preprocessing.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Load Dataset

Memuat dataset training dan testing dari file CSV.

In [2]:
train_path = "./data/train/train.csv"
test_path = "./data/test/test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("=== Train Dataset ===")
print(f"Shape: {df_train.shape}")
print(f"Columns: {df_train.columns.tolist()}")

print("\n=== Test Dataset ===")
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

=== Train Dataset ===
Shape: (49972, 4)
Columns: ['Body ID', 'articleBody', 'Headline', 'Stance']

=== Test Dataset ===
Shape: (25413, 3)
Columns: ['Body ID', 'articleBody', 'Headline']


# Define Preprocessing Function

Fungsi `clean_text()` melakukan pembersihan teks minimal yang diperlukan sebelum tokenisasi oleh Longformer. Preprocessing ini hanya menangani noise formatting (escape characters, whitespace berlebih, control characters) tanpa mengubah konten linguistik teks.

Langkah-langkah yang dilakukan:
1. Konversi input ke string
2. Ganti escape characters (`\n`, `\r`, `\t`) dengan spasi
3. Normalisasi kutipan ganda yang terduplikasi
4. Hapus invisible/control characters
5. Normalisasi whitespace berlebih
6. Strip leading dan trailing whitespace

Langkah-langkah yang sengaja tidak dilakukan:
- Tidak melakukan lowercasing
- Tidak menghapus tanda baca
- Tidak menghapus angka
- Tidak menghapus stopwords
- Tidak melakukan stemming
- Tidak melakukan lemmatization

In [3]:
def clean_text(text):
    """
    Minimal text cleaning for Longformer preprocessing.

    Handles formatting noise without altering linguistic content.
    Preserves original casing, punctuation, numbers, and stopwords.

    Parameters:
        text: Input text (any type, will be converted to str)

    Returns:
        str: Cleaned text
    """
    # Step 1: Convert to string
    text = str(text)

    # Step 2: Replace escape characters with spaces
    text = text.replace('\\n', ' ')
    text = text.replace('\\r', ' ')
    text = text.replace('\\t', ' ')

    # Step 3: Normalize duplicated quotes ("" -> ")
    text = re.sub(r'"{2,}', '"', text)

    # Step 4: Remove invisible/control characters (keep printable + whitespace)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)

    # Step 5: Normalize multiple whitespace to single space
    text = re.sub(r'\s+', ' ', text)

    # Step 6: Strip leading and trailing whitespace
    text = text.strip()

    return text

# Apply Preprocessing

Menerapkan fungsi `clean_text()` pada kolom Headline dan articleBody untuk dataset training dan testing.

In [4]:
# Apply to training dataset
print("Preprocessing training dataset...")
df_train['Headline'] = df_train['Headline'].apply(clean_text)
df_train['articleBody'] = df_train['articleBody'].apply(clean_text)

# Apply to testing dataset
print("Preprocessing testing dataset...")
df_test['Headline'] = df_test['Headline'].apply(clean_text)
df_test['articleBody'] = df_test['articleBody'].apply(clean_text)

print("Preprocessing complete.")

Preprocessing training dataset...
Preprocessing testing dataset...
Preprocessing complete.


# Before vs After Inspection

Menampilkan 5 sampel acak untuk memverifikasi hasil preprocessing secara manual. Perbandingan dilakukan dengan memuat ulang data asli dan membandingkannya dengan data yang sudah diproses.

In [5]:
# Reload original data for comparison
df_train_original = pd.read_csv(train_path)

# Select 5 random indices
np.random.seed(42)
sample_indices = np.random.choice(df_train.index, size=5, replace=False)

for idx in sample_indices:
    print("=" * 80)
    print(f"Sample index: {idx}")
    print()
    print(f"Original Headline:  {repr(df_train_original.loc[idx, 'Headline'])}")
    print(f"Cleaned Headline:   {repr(df_train.loc[idx, 'Headline'])}")
    print()
    original_body = str(df_train_original.loc[idx, 'articleBody'])[:300]
    cleaned_body = str(df_train.loc[idx, 'articleBody'])[:300]
    print(f"Original Article (first 300 chars):  {repr(original_body)}")
    print(f"Cleaned Article (first 300 chars):   {repr(cleaned_body)}")
    print()

print("=" * 80)

Sample index: 17851

Original Headline:  "Mum faces real-life Sophie's Choice – as she tries to sell her son to fund daughter's care"
Cleaned Headline:   "Mum faces real-life Sophie's Choice – as she tries to sell her son to fund daughter's care"

Original Article (first 300 chars):  '(Mashable) Reports that Islamic State militants in Mosul have contracted Ebola swirled though Iraqi media sources on Wednesday. World Health Organization officials said they haven’t confirmed the cases, but the organization has reached out to offer assistance.\n\nThree outlets reported that Ebola show'
Cleaned Article (first 300 chars):   '(Mashable) Reports that Islamic State militants in Mosul have contracted Ebola swirled though Iraqi media sources on Wednesday. World Health Organization officials said they haven’t confirmed the cases, but the organization has reached out to offer assistance. Three outlets reported that Ebola showe'

Sample index: 6400

Original Headline:  'Dog Found Abandoned With Sui

# Quality Check

Memeriksa kualitas data setelah preprocessing: missing values, headline kosong, dan article body kosong.

In [6]:
def quality_check(df, name):
    print(f"=== Quality Check: {name} ===")

    # Missing values
    missing = df.isnull().sum()
    print(f"\nMissing values:")
    print(missing)

    # Empty headlines (after stripping)
    empty_headlines = (df['Headline'].astype(str).str.strip() == '').sum()
    print(f"\nEmpty headlines: {empty_headlines}")

    # Empty article bodies (after stripping)
    empty_bodies = (df['articleBody'].astype(str).str.strip() == '').sum()
    print(f"Empty article bodies: {empty_bodies}")

    # Summary statistics
    print(f"\nHeadline length stats (chars):")
    print(df['Headline'].astype(str).str.len().describe())

    print(f"\nArticle body length stats (chars):")
    print(df['articleBody'].astype(str).str.len().describe())

    print()

quality_check(df_train, "Train")
quality_check(df_test, "Test")

=== Quality Check: Train ===

Missing values:
Body ID        0
articleBody    0
Headline       0
Stance         0
dtype: int64

Empty headlines: 0
Empty article bodies: 0

Headline length stats (chars):
count    49972.000000
mean        69.348135
std         24.818403
min          9.000000
25%         54.000000
50%         65.000000
75%         79.000000
max        225.000000
Name: Headline, dtype: float64

Article body length stats (chars):
count    49972.000000
mean      2196.795025
std       1668.599460
min         38.000000
25%       1168.000000
50%       1814.000000
75%       2750.000000
max      27451.000000
Name: articleBody, dtype: float64

=== Quality Check: Test ===

Missing values:
Body ID        0
articleBody    0
Headline       0
dtype: int64

Empty headlines: 0
Empty article bodies: 0

Headline length stats (chars):
count    25413.000000
mean        69.777673
std         26.133689
min          9.000000
25%         54.000000
50%         67.000000
75%         80.000000
max 

# Save Processed Dataset

Menyimpan dataset yang sudah dipreprocessing ke folder `./data/processed/`.

In [7]:
output_dir = Path("./data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_output = output_dir / "train_processed.csv"
test_output = output_dir / "test_processed.csv"

df_train.to_csv(train_output, index=False)
df_test.to_csv(test_output, index=False)

print(f"Train saved to: {train_output}")
print(f"Test saved to: {test_output}")
print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train saved to: data\processed\train_processed.csv
Test saved to: data\processed\test_processed.csv
Train shape: (49972, 4)
Test shape: (25413, 3)


# Summary

## Preprocessing Steps Applied

1. **String conversion** - Konversi semua input ke tipe string untuk konsistensi
2. **Escape character replacement** - Mengganti `\n`, `\r`, `\t` dengan spasi untuk menghilangkan formatting artifacts
3. **Duplicate quote normalization** - Menormalisasi kutipan ganda berulang (`""` menjadi `"`)
4. **Control character removal** - Menghapus invisible/control characters yang tidak tercetak
5. **Whitespace normalization** - Menggabungkan multiple whitespace menjadi satu spasi
6. **Whitespace stripping** - Menghapus spasi di awal dan akhir teks

## Preprocessing Steps Intentionally Not Applied

1. **Lowercasing** - Tidak dilakukan karena Longformer membutuhkan format teks asli untuk memahami konteks (huruf kapital bisa memberikan informasi penting seperti nama, akronim, dan penekanan)
2. **Punctuation removal** - Tidak dilakukan karena tanda baca mempengaruhi makna dan tokenisasi oleh Longformer
3. **Number removal** - Tidak dilakukan karena angka bisa menjadi informasi penting dalam berita
4. **Stopword removal** - Tidak dilakukan karena Longformer menggunakan attention mechanism yang sudah menangani stopwords secara implisit
5. **Stemming** - Tidak dilakukan karena bisa mengubah makna kata dan mengurangi informasi morfologis
6. **Lemmatization** - Tidak dilakukan karena alasan serupa dengan stemming

Pendekatan ini mempertahankan teks sealami mungkin agar Longformer dapat memanfaatkan representasi kontekstualnya secara optimal.